# Geography Chatbot with LangGraph

A small command-line style chatbot for students. It answers geography questions and politely declines other topics.

Before running the notebook, add an API key named `OPENAI_API_KEY` to Colab Secrets (the key icon in the left sidebar) and enable notebook access to that secret.

## 1. Install the required packages

Run this cell once in your Colab runtime.

In [ ]:
!pip -q install langgraph langchain langchain-openai

## 2. Load your API key from Colab Secrets

The key is read at runtime and stored only in this runtime environment. Never paste it into a code cell or save it in the notebook.

In [ ]:
import os
from google.colab import userdata

# Read the key from Colab Secrets and make it available to LangChain.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 3. Define the chatbot graph

The graph has one `chatbot` node. `add_messages` combines new messages with prior turns. An in-memory checkpointer remembers the conversation for the configured thread.

In [ ]:
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# Keep the chatbot focused on geography and politely redirect other questions.
SYSTEM = SystemMessage(
    content="You are a helpful chatbot for students. Answer questions about geography only. If asked about another topic, politely explain that you can help with geography questions instead."
)

# add_messages appends new turns to the message state.
class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

model = init_chat_model("openai:gpt-6-luna")

# Return the assistant reply as the node's only state update.
def chatbot(state: ChatState):
    reply = model.invoke([SYSTEM] + state["messages"])
    return {"messages": [reply]}

builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile(checkpointer=InMemorySaver())

## 4. Chat with the bot

Enter a geography question and press Return. Type `stop` to end the chat. The in-memory checkpointer keeps the conversation for this thread, so the loop does not need a separate history variable.

In [ ]:
config = {"configurable": {"thread_id": "1"}}

while True:
    text = input("You (type 'stop' to finish): ")

    if text.strip().lower() == "stop":
        print("Goodbye!")
        break

    result = graph.invoke({"messages": [HumanMessage(text)]}, config)
    print("Bot:", result["messages"][-1].content)